# Ensemble Analysis: Weight Optimisation and Error Diagnosis

The current ensemble achieves 0.95 on the leaderboard. This notebook investigates:

1. What does each model's confusion matrix look like in isolation?
2. Which genres does each model handle best and worst?
3. A simple grid search over ensemble weights to confirm the chosen weights are near-optimal on the validation set
4. Which genres the ensemble still gets wrong (candidates for future improvement)

This is a diagnostic notebook, not a training notebook.

In [ ]:
import random
import itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import torchvision.models as models
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
from sklearn.metrics import confusion_matrix

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DATA_ROOT = Path('/teamspace/studios/this_studio/data/messy_mashup')
CHKPT_DIR = Path('/teamspace/studios/this_studio/new_chkpt')

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for g, i in GENRE_TO_IDX.items()}

SR_CNN = 22050
SR_AST = 16000
DURATION = 5.0
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## Model Definitions

Each architecture must exactly mirror the training code so checkpoint weights load without key mismatches.

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.LeakyReLU(0.1),
                nn.MaxPool2d(2)
            )
        self.features = nn.Sequential(
            conv_block(1, 32),
            conv_block(32, 64),
            conv_block(64, 128),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.AdaptiveAvgPool2d(1),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


class AudioResNet34(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.resnet = models.resnet34(weights=None)
        self.resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)

    def forward(self, x):
        return self.resnet(x)


class AudioEfficientNetB0(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.model = models.efficientnet_b0(weights=None)
        self.model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, num_classes)

    def forward(self, x):
        return self.model(x)

In [ ]:
def load_model(model, checkpoint_path):
    state = torch.load(checkpoint_path, map_location=DEVICE)
    # Strip DataParallel prefix if present
    if any(k.startswith('module.') for k in state.keys()):
        state = {k.replace('module.', '', 1): v for k, v in state.items()}
    model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()
    return model

cnn_model    = load_model(CustomCNN(),        CHKPT_DIR / 'CustomCNN_best.pth')
resnet_model = load_model(AudioResNet34(),    CHKPT_DIR / 'messy-mashup_checkpoints_checkpoints_AudioResNet34_best.pth')
effnet_model = load_model(AudioEfficientNetB0(), CHKPT_DIR / 'EfficientNetB0_best.pth')
print('CNN models loaded.')

In [ ]:
from transformers import ASTForAudioClassification, AutoFeatureExtractor

AST_CKPT = CHKPT_DIR / 'messy-mashup_checkpoints_ast_best_phase2.pth'
ast_extractor = AutoFeatureExtractor.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')
ast_model = ASTForAudioClassification.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593',
    num_labels=10,
    ignore_mismatched_sizes=True
)
ast_state = torch.load(AST_CKPT, map_location=DEVICE)
if any(k.startswith('module.') for k in ast_state.keys()):
    ast_state = {k.replace('module.', '', 1): v for k, v in ast_state.items()}
ast_model.load_state_dict(ast_state)
ast_model.to(DEVICE)
ast_model.eval()
print('AST loaded.')

## Validation Dataset and Inference

In [ ]:
class ValDataset(Dataset):
    """Returns raw waveform (mono, 22050 Hz) and label. Mel transform applied in collate."""
    def __init__(self, root, genres, genre_to_idx, sr=SR_CNN, duration=DURATION):
        self.sr = sr
        self.n_samples = int(sr * duration)
        self.samples = []
        self.labels = []
        for genre in genres:
            genre_dir = root / 'val' / genre
            if not genre_dir.exists():
                continue
            for f in sorted(genre_dir.glob('*.wav')):
                try:
                    waveform, file_sr = torchaudio.load(str(f))
                    if file_sr != sr:
                        waveform = T.Resample(file_sr, sr)(waveform)
                    waveform = waveform.mean(dim=0)
                    self.samples.append(waveform)
                    self.labels.append(genre_to_idx[genre])
                except Exception:
                    pass

    def _crop_centre(self, waveform):
        n = waveform.shape[0]
        if n >= self.n_samples:
            start = (n - self.n_samples) // 2
            return waveform[start:start + self.n_samples]
        return torch.nn.functional.pad(waveform, (0, self.n_samples - n))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self._crop_centre(self.samples[idx]), self.labels[idx]

val_dataset = ValDataset(DATA_ROOT, GENRES, GENRE_TO_IDX)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
print(f'Validation samples: {len(val_dataset)}')

In [ ]:
mel_transform = nn.Sequential(
    T.MelSpectrogram(sample_rate=SR_CNN, n_fft=1024, hop_length=512,
                     n_mels=128, f_min=20, f_max=8000),
    T.AmplitudeToDB(top_db=80)
).to(DEVICE)

def get_cnn_probs(model, loader):
    all_probs, all_labels = [], []
    with torch.no_grad():
        for waveform, labels in loader:
            waveform = waveform.to(DEVICE)
            mel = mel_transform(waveform).unsqueeze(1)  # (B, 1, n_mels, T)
            with autocast():
                logits = model(mel)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_labels.extend(labels.numpy())
    return np.concatenate(all_probs), np.array(all_labels)

print('Getting CNN model probabilities...')
cnn_probs,    labels = get_cnn_probs(cnn_model,    val_loader)
resnet_probs, _      = get_cnn_probs(resnet_model, val_loader)
effnet_probs, _      = get_cnn_probs(effnet_model, val_loader)
print('Done.')

In [ ]:
# AST probabilities using its own feature extractor and resampling
def get_ast_probs(model, extractor, dataset, sr_ast=SR_AST):
    resample = T.Resample(SR_CNN, sr_ast)
    all_probs = []
    with torch.no_grad():
        for i in range(len(dataset)):
            waveform, _ = dataset[i]  # (n_samples,) at SR_CNN
            wav_16k = resample(waveform.unsqueeze(0)).squeeze(0).numpy()
            inputs = extractor(wav_16k, sampling_rate=sr_ast, return_tensors='pt')
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            with autocast():
                out = model(**inputs)
            probs = torch.softmax(out.logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            if i % 50 == 0:
                print(f'  AST: {i}/{len(dataset)}')
    return np.concatenate(all_probs)

print('Getting AST probabilities (this is slower, no batching)...')
ast_probs = get_ast_probs(ast_model, ast_extractor, val_dataset)
print('Done.')

## Per-model Accuracy and Confusion Matrices

In [ ]:
def plot_cm(cm, title, ax):
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(10))
    ax.set_xticklabels([g[:3].capitalize() for g in GENRES], rotation=45, ha='right')
    ax.set_yticks(range(10))
    ax.set_yticklabels([g[:3].capitalize() for g in GENRES])
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    for i in range(10):
        for j in range(10):
            ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                    fontsize=6, color='white' if cm_norm[i,j] > 0.5 else 'black')
    return im

model_probs = {
    'CustomCNN':          cnn_probs,
    'AudioResNet34':      resnet_probs,
    'EfficientNetB0':     effnet_probs,
    'AST (fine-tuned)':   ast_probs,
}

fig, axes = plt.subplots(2, 2, figsize=(18, 15))
axes = axes.flatten()

for ax, (name, probs) in zip(axes, model_probs.items()):
    preds = probs.argmax(axis=1)
    acc = (preds == labels).mean()
    cm = confusion_matrix(labels, preds)
    plot_cm(cm, f'{name} (Acc: {acc:.3f})', ax)

plt.suptitle('Per-model confusion matrices (normalised, validation set)', fontsize=13)
plt.tight_layout()
plt.savefig('per_model_confusion_matrices.png', dpi=130, bbox_inches='tight')
plt.show()

## Ensemble Weight Grid Search

We search over a coarse grid of weights for the four models. Weights are constrained to sum to 1. The current configuration is AST=0.40, ResNet=0.20, EfficientNet=0.25, CNN=0.15.

In [ ]:
# Coarse grid search: each weight in {0.05, 0.10, ..., 0.55}
step = 0.05
weight_options = np.arange(0.05, 0.60, step).round(2)

best_acc = 0.0
best_weights = None
results = []

# Enumerate combinations that sum to 1 (within floating point tolerance)
for w_ast, w_res, w_eff in itertools.product(weight_options, repeat=3):
    w_cnn = round(1.0 - w_ast - w_res - w_eff, 4)
    if w_cnn < 0 or w_cnn > weight_options.max():
        continue
    ensemble = (w_ast * ast_probs +
                w_res * resnet_probs +
                w_eff * effnet_probs +
                w_cnn * cnn_probs)
    preds = ensemble.argmax(axis=1)
    acc = (preds == labels).mean()
    results.append({'w_ast': w_ast, 'w_res': w_res, 'w_eff': w_eff, 'w_cnn': w_cnn, 'val_acc': acc})
    if acc > best_acc:
        best_acc = acc
        best_weights = (w_ast, w_res, w_eff, w_cnn)

df_grid = pd.DataFrame(results).sort_values('val_acc', ascending=False)
print(f'Best val accuracy: {best_acc:.4f}')
print(f'Best weights: AST={best_weights[0]}, ResNet={best_weights[1]}, EffNet={best_weights[2]}, CNN={best_weights[3]}')
print(f'\nTop 10 configurations:')
print(df_grid.head(10).to_string(index=False))

## Ensemble Error Analysis

In [ ]:
# Final ensemble with current weights
ensemble_probs = (0.40 * ast_probs + 0.20 * resnet_probs +
                  0.25 * effnet_probs + 0.15 * cnn_probs)
ensemble_preds = ensemble_probs.argmax(axis=1)
ensemble_acc = (ensemble_preds == labels).mean()
print(f'Ensemble accuracy (current weights): {ensemble_acc:.4f}')

cm = confusion_matrix(labels, ensemble_preds)

fig, ax = plt.subplots(figsize=(10, 8))
plot_cm(cm, f'Ensemble Confusion Matrix (val acc = {ensemble_acc:.4f})', ax)
plt.tight_layout()
plt.savefig('ensemble_confusion_matrix.png', dpi=150)
plt.show()

# Per-class accuracy
per_class = {}
for i, genre in enumerate(GENRES):
    mask = labels == i
    per_class[genre] = (ensemble_preds[mask] == labels[mask]).mean() if mask.sum() > 0 else 0.0

df_per_class = pd.DataFrame(list(per_class.items()), columns=['genre', 'accuracy'])
df_per_class = df_per_class.sort_values('accuracy')

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['red' if a < 0.90 else 'steelblue' for a in df_per_class['accuracy']]
ax.barh(df_per_class['genre'], df_per_class['accuracy'], color=colors)
ax.set_xlabel('Per-class Accuracy')
ax.set_title('Ensemble per-genre accuracy (red = below 90%)')
ax.axvline(x=0.90, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('per_genre_accuracy.png', dpi=150)
plt.show()

print('Weakest genres (target for future improvement):')
print(df_per_class.head(3).to_string(index=False))

## Takeaways

- The grid search validates whether the current weights are near the optimum or whether there is a clearly better configuration on the validation set.
- The per-genre accuracy plot highlights which classes to focus on. Genres that are consistently confused (e.g. rock vs metal, disco vs pop) could benefit from:
  - Hard negative mining: oversample misclassified examples
  - Specialised fine-tuning on confusable pairs
  - Adding a genre-specific post-processing step